# 01 — Data Validation

Runs the data contract defined in `configs/data.yaml` against the raw dataset via `BNPLDataValidator`, using the same `TrainingInputSchema` the training pipeline uses. See `docs/data_contract.md` for the full contract and the leakage audit.

In [1]:
from bnpl_credit_risk.data.cleaning import BNPLDataCleaner
from bnpl_credit_risk.data.loaders import BNPLDataLoader
from bnpl_credit_risk.data.schemas import inference_input_schema, training_input_schema
from bnpl_credit_risk.data.validation import BNPLDataValidator
from bnpl_credit_risk.settings import get_settings, load_config

settings = get_settings()
config = load_config()
df = BNPLDataCleaner(config.data).clean(BNPLDataLoader(settings, config.data).load_raw())

2026-07-13 08:35:57.403 | INFO     | bnpl_credit_risk.data.loaders:load_raw:37 - Loaded raw dataset path=/Users/surelmanda/BNPL-Credit-Risk/data/raw/BNPL_CreditRisk_Dataset.csv rows=10345 columns=17


2026-07-13 08:35:57.421 | INFO     | bnpl_credit_risk.data.cleaning:clean:45 - Cleaned dataframe rows=10345 columns=17


## Training schema (target required)

In [2]:
schema = training_input_schema(config.data)
validator = BNPLDataValidator(schema, config.data.validation.strategy)
result = validator.validate(df)
result.report

2026-07-13 08:35:57.447 | INFO     | bnpl_credit_risk.data.validation:validate:122 - Validation complete strategy=strict valid_rows=10345 invalid_rows=0


{'n_valid_rows': 10345,
 'n_invalid_rows': 0,
 'structural_violations': [],
 'row_issue_counts': {}}

## Inference schema (target absent)

`inference_input_schema` never requires `default_flag` — this is what `predict-batch` validates against.

In [3]:
inference_schema = inference_input_schema(config.data)
inference_schema.target_column, inference_schema.required_columns[:5]

(None, ['user_id', 'age', 'employment_type', 'monthly_income', 'credit_score'])